In [2]:
import json
from pathlib import Path

import nibabel as nib
import numpy as np
from scipy import ndimage
from skimage.segmentation import clear_border
from skimage.measure import label
from skimage.feature import blob_log
from tqdm.notebook import tqdm

MIN_LUNG_HU = -950
MAX_LUNG_HU = -650
MIN_NODULE_HU = -150
MAX_NODULE_HU = 200

In [3]:
json_path = Path("/home/chest_ct/code/data/rexgrounding-ct/dataset_2d_filtered.json")

volume_root = Path("/home/chest_ct/code/data/data_volumes/dataset/train_fixed")
mask_root = Path("/home/chest_ct/code/data/segmentations/segmentations")

output_root = Path("/home/chest_ct/code/data/lung_masked_cts")
output_root.mkdir(parents=True, exist_ok=True)

In [ ]:
# -----------------------------
# Helper functions
# -----------------------------
def keep_largest_components(binary_mask, n=2):
    labeled, num = label(binary_mask, return_num=True, connectivity=1)

    if num == 0:
        return np.zeros_like(binary_mask, dtype=bool)

    component_sizes = np.bincount(labeled.ravel())
    component_sizes[0] = 0

    largest_labels = np.argsort(component_sizes)[-n:]

    return np.isin(labeled, largest_labels)


def find_ct_path(filename):
    matches = list(volume_root.rglob(filename))

    if len(matches) == 0:
        return None

    return matches[0]


def lung_preprocess(ct):
    lung_candidate = (ct >= MIN_LUNG_HU) & (ct <= MAX_LUNG_HU)

    mask_clear = np.zeros_like(lung_candidate, dtype=bool)

    for k in range(lung_candidate.shape[2]):
        mask_clear[:, :, k] = clear_border(lung_candidate[:, :, k])

    lungs_only = keep_largest_components(mask_clear, n=2)

    structure = ndimage.generate_binary_structure(3, 2)

    mask_closed = ndimage.binary_closing(
        lungs_only,
        structure=structure,
        iterations=2
    )

    mask_filled = ndimage.binary_fill_holes(mask_closed)

    lungs_final = ndimage.binary_closing(
        mask_filled,
        structure=structure,
        iterations=2
    )

    lungs_final = ndimage.binary_fill_holes(lungs_final)

    ct_lung_only = ct.copy()
    ct_lung_only[~lungs_final] = -1000

    return ct_lung_only


def nlog_blob_detection_3d(ct_lung_only, hu_min=-150, hu_max=200):
    candidate = (ct_lung_only >= hu_min) & (ct_lung_only <= hu_max)

    blob_input = ct_lung_only.copy()
    blob_input[~candidate] = 0

    blobs = blob_log(
        blob_input,
        min_sigma=1,
        max_sigma=8,
        num_sigma=10,
        threshold=0.05
    )

    blobs[:, 3] = blobs[:, 3] * np.sqrt(3)

    return blobs, candidate

# -----------------------------
# Load filenames
# -----------------------------
with open(json_path, "r") as f:
    dataset = json.load(f)

ct_names = dataset["train"]

print("Number of CTs:", len(ct_names))

Number of CTs: 271


In [ ]:
# -----------------------------
# Run for all CTs
# -----------------------------
missing_ct = []
missing_mask = []
failed = []

for filename in tqdm(ct_names):
    try:
        ct_path = find_ct_path(filename)
        mask_path = mask_root / filename

        if ct_path is None:
            missing_ct.append(filename)
            continue

        if not mask_path.exists():
            missing_mask.append(filename)
            continue

        ct_img = nib.load(str(ct_path))
        ct = ct_img.get_fdata()

        gt_img = nib.load(str(mask_path))
        gt_mask = gt_img.get_fdata()

        if gt_mask.ndim == 4:
            gt_mask = gt_mask[0]

        ct_lung_only = lung_preprocess(ct)

        blobs, nodule_candidate = nlog_blob_detection_3d(ct_lung_only)
        print(f"Detected {len(blobs)} blobs")

    except Exception as e:
        failed.append((filename, str(e)))


print("Done")
print("Missing CTs:", len(missing_ct))
print("Missing masks:", len(missing_mask))
print("Failed:", len(failed))

if missing_ct:
    print("Example missing CT:", missing_ct[:5])

if missing_mask:
    print("Example missing mask:", missing_mask[:5])

if failed:
    print("Example failed:", failed[:5])